# Upload Clustering Outputs to S3

Uploads `louvain_clusters.txt` from the local `output/` snapshot/query folder to the OpenAlex S3 clustering destination.
Set `SNAPSHOT` and `QUERY` in the configuration cell below before running.

## 1. Import Required Libraries

In [1]:
import boto3
from pathlib import Path

## 2. Set Configuration Variables

Edit `SNAPSHOT` and `QUERY` to match the dataset you want to upload.

In [ ]:
# -- Change these two values before running ------------------------------------
SNAPSHOT = "2026-06-30"
QUERY    = "sustainability"
# ------------------------------------------------------------------------------

S3_BUCKET   = "openalex-results"
FILENAME    = "louvain_clusters.txt"
S3_KEY      = (
    f"snapshot_{SNAPSHOT}/queries/{QUERY}/network/clustering/"
    f"{FILENAME}/{FILENAME}"
)
BASE_OUTPUT = Path("output")
local_file  = BASE_OUTPUT / f"snapshot_{SNAPSHOT}" / QUERY / FILENAME

print(f"Snapshot     : {SNAPSHOT}")
print(f"Query        : {QUERY}")
print(f"Local source : {local_file}")
print(f"S3 dest      : s3://{S3_BUCKET}/{S3_KEY}")

Local source : output/planetary-health/version3/louvain_clusters.txt
S3 dest      : s3://openalex-outputs/cwts/planetary-health/network_assets/version3/louvain_clusters.txt/louvain_clusters.txt


## 3. Verify Local File Exists

In [3]:
import pandas as pd

EXPECTED_COLUMNS = ["node_id", "micro_cluster", "meso_cluster", "macro_cluster"]

# --- Check file exists
if not local_file.exists():
    raise FileNotFoundError(
        f"Output file not found: {local_file}\n"
        "Run parallel_louvain.ipynb first to generate the clustering output."
    )

size_mb = local_file.stat().st_size / (1024 ** 2)
print(f"✓ File found: {local_file}")
print(f"  Size: {size_mb:.2f} MB")

# --- Check file is not empty
if local_file.stat().st_size == 0:
    raise ValueError(f"Output file is empty: {local_file}")

# --- Check column names match expected schema
df_head = pd.read_csv(local_file, sep="\t", nrows=5)
actual_columns = list(df_head.columns)

if actual_columns != EXPECTED_COLUMNS:
    raise ValueError(
        f"Column mismatch in {local_file.name}:\n"
        f"  Expected : {EXPECTED_COLUMNS}\n"
        f"  Found    : {actual_columns}"
    )

print(f"✓ Columns verified: {actual_columns}")
print(f"  Row count (preview): {len(df_head)}")

✓ File found: output/planetary-health/version3/louvain_clusters.txt
  Size: 0.05 MB
✓ Columns verified: ['node_id', 'micro_cluster', 'meso_cluster', 'macro_cluster']
  Row count (preview): 5


## 4. Upload File to S3

Uses the default AWS credential chain (environment variables, `~/.aws/credentials`, or instance profile).

In [4]:
s3 = boto3.client("s3")

file_size = local_file.stat().st_size
print(f"Uploading {local_file.name} ({file_size / (1024**2):.2f} MB) ...")

# Use multipart upload via TransferConfig for large files
from boto3.s3.transfer import TransferConfig

config = TransferConfig(
    multipart_threshold=64 * 1024 * 1024,   # 64 MB
    multipart_chunksize=64 * 1024 * 1024,   # 64 MB per part
)

s3.upload_file(
    Filename=str(local_file),
    Bucket=S3_BUCKET,
    Key=S3_KEY,
    Config=config,
)

print(f"\n✓ Upload complete.")
print(f"  s3://{S3_BUCKET}/{S3_KEY}")

Uploading louvain_clusters.txt (0.05 MB) ...

✓ Upload complete.
  s3://openalex-outputs/cwts/planetary-health/network_assets/version3/louvain_clusters.txt/louvain_clusters.txt


## 5. Verify Upload

In [5]:
response = s3.head_object(Bucket=S3_BUCKET, Key=S3_KEY)
s3_size_mb = response["ContentLength"] / (1024 ** 2)
last_modified = response["LastModified"]

print(f"✓ Verified on S3:")
print(f"  Key          : {S3_KEY}")
print(f"  Size         : {s3_size_mb:.2f} MB")
print(f"  Last modified: {last_modified}")

✓ Verified on S3:
  Key          : cwts/planetary-health/network_assets/version3/louvain_clusters.txt/louvain_clusters.txt
  Size         : 0.05 MB
  Last modified: 2026-07-05 05:01:47+00:00


## 6. Create and Upload Provenance Markdown

Collects run metadata (date, instance specs, git info, output stats) and writes
`louvain_clusters.md` alongside the local output file, then uploads it to S3.

In [ ]:
import datetime, os, platform, subprocess, sys, shutil, urllib.request
from importlib.metadata import version as _pkg_version

# -- Manual input: fill in only if auto-detection returns "N/A" ---------------
INSTANCE_NAME = ""
# ------------------------------------------------------------------------------


def _run(cmd):
    """Run a subprocess and return stripped stdout, or 'N/A' on failure."""
    try:
        return subprocess.check_output(cmd, stderr=subprocess.DEVNULL, text=True).strip()
    except Exception:
        return "N/A"


def _imds(path, token):
    """Fetch a value from EC2 Instance Metadata Service (IMDSv2)."""
    try:
        req = urllib.request.Request(
            f"http://169.254.169.254/latest/meta-data/{path}",
            headers={"X-aws-ec2-metadata-token": token},
        )
        with urllib.request.urlopen(req, timeout=2) as r:
            return r.read().decode()
    except Exception:
        return "N/A"


_token = None
try:
    _tok_req = urllib.request.Request(
        "http://169.254.169.254/latest/api/token",
        method="PUT",
        headers={"X-aws-ec2-metadata-token-ttl-seconds": "21600"},
    )
    with urllib.request.urlopen(_tok_req, timeout=2) as r:
        _token = r.read().decode()
except Exception:
    pass

instance_id = _imds("instance-id", _token) if _token else "N/A"
instance_type = _imds("instance-type", _token) if _token else "N/A"
availability_zone = _imds("placement/availability-zone", _token) if _token else "N/A"
ami_id = _imds("ami-id", _token) if _token else "N/A"

if not INSTANCE_NAME:
    try:
        region = availability_zone[:-1] if availability_zone != "N/A" else None
        _ec2 = boto3.client("ec2", **({"region_name": region} if region else {}))
        _tags = _ec2.describe_tags(Filters=[
            {"Name": "resource-id", "Values": [instance_id]},
            {"Name": "key", "Values": ["Name"]},
        ]).get("Tags", [])
        INSTANCE_NAME = _tags[0]["Value"] if _tags else platform.node()
    except Exception:
        INSTANCE_NAME = platform.node()

ram_total = "N/A"
try:
    with open("/proc/meminfo") as _f:
        for _line in _f:
            if _line.startswith("MemTotal:"):
                ram_total = f"{int(_line.split()[1]) / (1024 ** 2):.1f} GB"
                break
except Exception:
    pass

_disk = shutil.disk_usage("/")
disk_total = f"{_disk.total / (1024 ** 3):.1f} GB"
disk_used = f"{_disk.used / (1024 ** 3):.1f} GB"
disk_free = f"{_disk.free / (1024 ** 3):.1f} GB"

node_count = "N/A"
cluster_count = "N/A"
try:
    clusters = set()
    lines = 0
    with open(local_file) as fh:
        for row in fh:
            row = row.rstrip("\n")
            if not row:
                continue
            if row.startswith("node_id\t"):
                continue
            parts = row.split("\t")
            if len(parts) >= 2:
                clusters.add(parts[1])
            lines += 1
    node_count = f"{lines:,}"
    cluster_count = f"{len(clusters):,}"
except Exception:
    pass

python_version = sys.version.replace("\n", " ")
try:
    nk_version = _pkg_version("networkit")
except Exception:
    nk_version = "N/A"
try:
    pandas_version = _pkg_version("pandas")
except Exception:
    pandas_version = "N/A"
try:
    boto3_version = _pkg_version("boto3")
except Exception:
    boto3_version = getattr(boto3, "__version__", "N/A")

git_origin = _run(["git", "remote", "get-url", "origin"])
git_branch = _run(["git", "rev-parse", "--abbrev-ref", "HEAD"])
git_commit = _run(["git", "rev-parse", "HEAD"])

run_date = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
cwd = str(Path.cwd())
os_info = platform.platform()
hostname = platform.node()
file_size = f"{local_file.stat().st_size / (1024 ** 2):.2f} MB"

provenance_md = f"""\
# louvain_clusters.txt - Provenance

## Dataset
| Field    | Value       |
|----------|-------------|
| Snapshot | `{SNAPSHOT}` |
| Query    | `{QUERY}`   |

## Output File
| Field    | Value            |
|----------|------------------|
| Path     | `{local_file}`   |
| Size     | {file_size}      |
| Nodes    | {node_count}     |
| Clusters | {cluster_count}  |

## Run Environment
| Field             | Value                 |
|-------------------|-----------------------|
| Date (UTC)        | {run_date}            |
| Working directory | `{cwd}`               |
| Hostname          | {hostname}            |
| Instance name     | {INSTANCE_NAME}       |
| EC2 instance ID   | {instance_id}         |
| EC2 instance type | {instance_type}       |
| Availability zone | {availability_zone}   |
| AMI ID            | {ami_id}              |

## System Specs
| Field      | Value          |
|------------|----------------|
| OS         | {os_info}      |
| vCPUs      | {os.cpu_count()} |
| RAM        | {ram_total}    |
| Disk total | {disk_total}   |
| Disk used  | {disk_used}    |
| Disk free  | {disk_free}    |

## Software
| Package   | Version          |
|-----------|------------------|
| Python    | {python_version} |
| NetworKit | {nk_version}     |
| pandas    | {pandas_version} |
| boto3     | {boto3_version}  |

## Code Repository
| Field      | Value          |
|------------|----------------|
| Git origin | {git_origin}   |
| Branch     | {git_branch}   |
| Commit     | `{git_commit}` |
"""

provenance_path = local_file.parent / "louvain_clusters.md"
provenance_path.write_text(provenance_md)
print(f"✓ Provenance written locally : {provenance_path}")

provenance_s3_key = (
    f"snapshot_{SNAPSHOT}/queries/{QUERY}/network/clustering/louvain_clusters.md"
)
s3.put_object(
    Bucket=S3_BUCKET,
    Key=provenance_s3_key,
    Body=provenance_md.encode(),
    ContentType="text/markdown",
)
print(f"✓ Provenance uploaded to S3  : s3://{S3_BUCKET}/{provenance_s3_key}")
print()
print(provenance_md)

✓ README written locally : output/planetary-health/version3/louvain_clusters_README.md
✓ README uploaded to S3  : s3://openalex-outputs/cwts/planetary-health/network_assets/version3/louvain_clusters_README.md

# louvain_clusters.txt — Provenance

## Dataset
| Field   | Value       |
|---------|-------------|
| Query   | `planetary-health`   |
| Version | `version3` |

## Output File
| Field    | Value            |
|----------|------------------|
| Path     | `output/planetary-health/version3/louvain_clusters.txt`   |
| Size     | 0.05 MB      |
| Nodes    | 3,803     |
| Clusters | 113  |

## Run Environment
| Field             | Value                 |
|-------------------|-----------------------|
| Date (UTC)        | 2026-07-05 05:01:47 UTC            |
| Working directory | `/home/ubuntu/parallel_louvain`               |
| Hostname          | ip-172-31-38-110            |
| Instance name     | ec2_64vcpu_512ram_intelxion       |
| EC2 instance ID   | i-051f1498f046cb081         |
|